# VANGROVE #2 - Dataset Extraction

VANGROVE (Visual Analytics & Navigation for Geographic Regional Output & Variety Evaluation)

### Deskripsi Notebook
Notebook ini berfokus pada tahap ekstraksi dataset daun tanaman dari repositori Kaggle (hasil preprocessing dan cleaning dari notebook #1) ke dalam format terstruktur yang siap digunakan untuk proses pengolahan data lanjutan. Tahapan pada notebook ini mencakup proses pengunduhan dataset, konversi struktur folder gambar menjadi *Pandas DataFrame*, serta validasi dan pembersihan dasar data. Tahapan yang dilakukan pada notebook ini meliputi:

1. **Dataset Acquisition from Kaggle**  
   Mengunduh dataset citra daun tanaman dari repositori Kaggle menggunakan `kagglehub`

2. **Dataset Structure Extraction**  
   Mengekstrak struktur folder dataset gambar menjadi format tabular (*Pandas DataFrame*) yang berisi kategori tanaman, penyakit, dan path gambar

3. **Basic Data Validation & Cleaning**  
   Melakukan validasi dataset serta pembersihan dasar seperti pengecekan *missing values* dan duplikasi path gambar

4. **Base Dataset Export**  
   Menyimpan dataset dasar (`base_dataset.csv`) sebagai fondasi untuk tahap *Feature Engineering* dan *Data Preparation & Balancing* selanjutnya

**Output:**  
Tahap ini menghasilkan dataset dasar (`base_dataset.csv`) yang telah terstruktur dan tervalidasi untuk digunakan pada proses pengolahan data lanjutan.

In [10]:
# Mengunggah file kaggle.json untuk autentikasi Kaggle API
from google.colab import files
files.upload()

Saving kaggle.json to kaggle (1).json


{'kaggle (1).json': b'{"username":"michaelmunthe123","key":"93b2bf194567fb1094f3d6e5c06439ff"}'}

In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [13]:
# Menginstal library kagglehub versi terbaru
!pip install -q kagglehub

import kagglehub

# Mendownload dan mengekstrak dataset secara otomatis dari Kaggle
print("Sedang mendownload dataset dari Kaggle...")
path = kagglehub.dataset_download("pppiiiy/data-science-data")

print("✅ Dataset sudah siap! Lokasinya ada di:", path)

Sedang mendownload dataset dari Kaggle...
✅ Dataset sudah siap! Lokasinya ada di: /root/.cache/kagglehub/datasets/pppiiiy/data-science-data/versions/1


### Membaca Struktur Folder Menjadi DataFrame
Dataset yang diunduh berbentuk file gambar fisik yang tersusun dalam folder `Tanaman > Penyakit`. Kode di bawah ini akan menelusuri direktori tersebut dan mencatat setiap *path* gambar ke dalam tabel.

In [7]:
import os
import pandas as pd

# Path langsung menembak ke folder 'combined'
DATASET_PATH = "/root/.cache/kagglehub/datasets/pppiiiy/data-science-data/versions/1/combined"

file_paths = []

# Looping membaca folder tanaman dan penyakit
for plant_name in os.listdir(DATASET_PATH):
    plant_path = os.path.join(DATASET_PATH, plant_name)

    if os.path.isdir(plant_path):
        for disease_name in os.listdir(plant_path):
            disease_path = os.path.join(plant_path, disease_name)

            if os.path.isdir(disease_path):
                for img_file in os.listdir(disease_path):
                    if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                        file_paths.append({
                            "plant": plant_name,
                            "disease": disease_name,
                            "image_path": os.path.join(disease_path, img_file) # Path asli di Colab
                        })

# Ubah menjadi DataFrame
df_cleaned = pd.DataFrame(file_paths)

print(f"Berhasil membuat tabel DataFrame dengan {len(df_cleaned)} baris gambar!")
display(df_cleaned.head())

Berhasil membuat tabel DataFrame dengan 47516 baris gambar!


,plant,disease,image_path
0,tomato,target_spot,/root/.cache/kagglehub/datasets/pppiiiy/data-s...
1,tomato,target_spot,/root/.cache/kagglehub/datasets/pppiiiy/data-s...
2,tomato,target_spot,/root/.cache/kagglehub/datasets/pppiiiy/data-s...
3,tomato,target_spot,/root/.cache/kagglehub/datasets/pppiiiy/data-s...
4,tomato,target_spot,/root/.cache/kagglehub/datasets/pppiiiy/data-s...


In [14]:
# --- TAHAP CLEANING (PEMBERSIHAN DATA) ---

print("1. Mengecek Missing Values:")
print(df_cleaned.isnull().sum())
df_cleaned = df_cleaned.dropna() # Menghapus baris yang kosong

print("\n2. Mengecek dan Menghapus Duplikat:")
jumlah_awal = len(df_cleaned)
df_cleaned = df_cleaned.drop_duplicates(subset=['image_path']) # Hapus gambar yang path-nya sama
jumlah_akhir = len(df_cleaned)

print(f"Jumlah baris awal: {jumlah_awal}")
print(f"Jumlah baris setelah dibersihkan: {jumlah_akhir}")
print(f"Total duplikat yang dibuang: {jumlah_awal - jumlah_akhir}")

display(df_cleaned.head())

1. Mengecek Missing Values:
plant         0
disease       0
image_path    0
dtype: int64

2. Mengecek dan Menghapus Duplikat:
Jumlah baris awal: 47516
Jumlah baris setelah dibersihkan: 47516
Total duplikat yang dibuang: 0


,plant,disease,image_path
0,tomato,target_spot,/root/.cache/kagglehub/datasets/pppiiiy/data-s...
1,tomato,target_spot,/root/.cache/kagglehub/datasets/pppiiiy/data-s...
2,tomato,target_spot,/root/.cache/kagglehub/datasets/pppiiiy/data-s...
3,tomato,target_spot,/root/.cache/kagglehub/datasets/pppiiiy/data-s...
4,tomato,target_spot,/root/.cache/kagglehub/datasets/pppiiiy/data-s...


In [15]:
# Menyimpan dataset dasar yang sudah bersih ke dalam CSV
df_cleaned.to_csv("base_dataset.csv", index=False)

print("✅ File 'base_dataset.csv' berhasil disimpan dan siap digunakan untuk tahap Feature Engineering!")

✅ File 'base_dataset.csv' berhasil disimpan dan siap digunakan untuk tahap Feature Engineering!
